# GSE193531 — Step 2: Normalization and Per-Cohort Preprocessing

Loads the QC-filtered AnnData object, backs up raw counts, normalizes and
log1p-transforms the full gene set, flags highly variable genes (`seurat_v3`,
computed on raw counts), computes a per-cohort PCA/UMAP/Leiden clustering as a
sanity check (and as the input to unbiased marker discovery used by step 3),
and saves the normalized object.

**Input:**
- `data/processed/GSE193531/GSE193531_filtered.h5ad` (from notebook 01)
- `results/GSE193531/qc/GSE193531_cell_qc_filtered.csv`

**Output:**
- `data/processed/GSE193531/GSE193531_normalized.h5ad`
- `results/GSE193531/preprocessing/GSE193531_leiden_cluster_summary.csv`

One step here is a deliberate, documented judgment call, not a mechanical
computation: flagging specific Leiden clusters as non-plasma contaminants or
QC-caution (`cell_type_candidate`). Leiden clustering is stochastic — cluster
numbers from a fresh run will not match any numbers recorded in an earlier run
of this notebook, so no specific cluster IDs are hardcoded here. Instead, this
notebook ships a conservative default (no clusters flagged) and prints the
evidence (per-cluster top marker genes, QC metrics) needed to review and edit
`MYELOID_CLUSTERS` / `B_CELL_CLUSTERS` / `QC_CAUTION_CLUSTERS` below before
trusting `cell_type_candidate` for anything beyond the sanity-check role it
plays in step 3.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1

#### Directory layout and required files

In [ ]:
PROJECT = Path("../")
COHORT = "GSE193531"

PROCESSED_DIR = PROJECT / "data" / "processed" / COHORT
RESULTS_DIR = PROJECT / "results" / COHORT

(RESULTS_DIR / "preprocessing").mkdir(parents=True, exist_ok=True)

INPUT_H5AD = PROCESSED_DIR / f"{COHORT}_filtered.h5ad"
INPUT_QC_CSV = RESULTS_DIR / "qc" / f"{COHORT}_cell_qc_filtered.csv"

if not INPUT_H5AD.exists():
    raise FileNotFoundError(INPUT_H5AD)
if not INPUT_QC_CSV.exists():
    raise FileNotFoundError(INPUT_QC_CSV)

OUTPUT_H5AD = PROCESSED_DIR / f"{COHORT}_normalized.h5ad"

N_TOP_GENES = 3000
MIN_CELLS_FOR_BATCHED_HVG = 500

## Load Filtered Count Matrix and Re-Attach QC Columns

`sample_id`, `patient_id`, `clinical_stage_standard`, `n_genes`,
`doublet_score`, and `predicted_doublet` are already on `adata.obs` from
notebook 01; the remaining QC metrics (`n_counts`, `pct_mito`, `pct_ribo`,
`pct_hb`, `pct_ig`, `complexity`) are re-attached from the QC table rather
than recomputed.

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)

qc_filtered = pd.read_csv(INPUT_QC_CSV, index_col=0)

if not qc_filtered.index.equals(adata.obs_names):
    raise ValueError("QC table index does not match AnnData cell order. Do not proceed until this is resolved.")

extra_qc_cols = ["n_counts", "pct_mito", "pct_ribo", "pct_hb", "pct_ig", "complexity"]
adata.obs = adata.obs.join(qc_filtered[extra_qc_cols])

print(adata)

## Preserve Raw Counts, Normalize, Log-Transform

Raw counts are copied to `adata.layers["counts"]` before any transformation
(a sanity check confirms `.X` is still integer-valued at this point). Then
library-size normalization (target sum 1e4) and `log1p`.

In [ ]:
sample = adata.X[:500].toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X[:500])
if not np.allclose(sample, np.round(sample)):
    raise ValueError("adata.X does not look like raw counts (non-integer values found). Aborting.")

adata.layers["counts"] = adata.X.copy()
print('Raw counts backed up to adata.layers["counts"]')

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print("Normalized and log1p-transformed. adata.X now holds log-normalized values.")

## Highly Variable Genes

`seurat_v3` fits its own mean-variance model and needs **raw counts** —
pointed explicitly at `adata.layers["counts"]`, not `adata.X`. Genes are
selected per sample (`batch_key="sample_id"`) when every sample has enough
cells; otherwise this falls back to a single cohort-wide fit. Genes are
flagged in `adata.var["highly_variable"]` only, **not subsetted** — the full
gene set stays available downstream.

In [ ]:
min_cells_per_sample = int(adata.obs["sample_id"].value_counts().min())
use_batch_key = min_cells_per_sample >= MIN_CELLS_FOR_BATCHED_HVG

hvg_kwargs = dict(n_top_genes=N_TOP_GENES, flavor="seurat_v3", layer="counts", subset=False)

try:
    sc.pp.highly_variable_genes(adata, batch_key="sample_id" if use_batch_key else None, **hvg_kwargs)
except ValueError as exc:
    print(f"Per-sample HVG fit failed ({exc!r}); falling back to a single cohort-wide fit.")
    sc.pp.highly_variable_genes(adata, batch_key=None, **hvg_kwargs)

n_hvg = int(adata.var["highly_variable"].sum())
batch_fit_status = "used" if use_batch_key else "skipped"
print(f"Smallest sample has {min_cells_per_sample:,} cells (batched fit {batch_fit_status}).")
print(f"Flagged {n_hvg:,} / {adata.n_vars:,} genes as highly variable (target {N_TOP_GENES:,}).")

## PCA, Neighbors, UMAP, Leiden Clustering (Per-Cohort Sanity Check)

PCA is computed on a **separate scaled copy** of the HVG subset — `adata.X`
itself is never overwritten. The PCA embedding is attached back onto the full
`adata` object, and neighbors/UMAP/Leiden are computed from it.

In [ ]:
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata_hvg, max_value=10)

n_comps = min(50, adata_hvg.n_vars - 1)
sc.tl.pca(adata_hvg, svd_solver="arpack", n_comps=n_comps)

adata.obsm["X_pca"] = adata_hvg.obsm["X_pca"]
adata.uns["pca"] = adata_hvg.uns["pca"]

n_pcs = min(30, n_comps)
sc.pp.neighbors(adata, n_pcs=n_pcs, use_rep="X_pca")
sc.tl.umap(adata, min_dist=0.5)
sc.tl.leiden(adata, resolution=1.0)

print(f"PCA: {n_comps} components on {adata_hvg.n_vars:,} HVGs")
print(f"Leiden clusters: {adata.obs['leiden'].nunique()}")

## UMAP Diagnostics

In [ ]:
sc.pl.umap(adata, color=["sample_id", "clinical_stage_standard", "leiden"], wspace=0.4, legend_loc="on data", legend_fontsize=6)
sc.pl.umap(adata, color=["n_counts", "pct_mito", "doublet_score", "complexity"], wspace=0.4, cmap="viridis")

marker_genes_umap = ["SDC1", "MZB1", "PRDM1", "IGHG1", "IGKC"]
available_markers = [g for g in marker_genes_umap if g in adata.var_names]
missing_markers = [g for g in marker_genes_umap if g not in adata.var_names]
if missing_markers:
    print("Markers not found in this cohort:", missing_markers)
if available_markers:
    sc.pl.umap(adata, color=available_markers, ncols=3, size=40, cmap="viridis")

## Cluster Annotation QC

Unbiased per-cluster marker discovery first (`rank_genes_groups`, kept in
`adata.uns` for step 3's DE cross-check), then a curated marker dotplot
including negative-control markers for the cell types most likely to be
confused with each other, then a sample-composition check before trusting any
single-sample-dominated cluster as biology rather than a batch artifact.

In [ ]:
display(adata.obs["leiden"].value_counts().sort_index())

sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
top_markers = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(10)
top_markers

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

### QC Metrics per Cluster

In [ ]:
qc_metrics = ["n_counts", "n_genes", "pct_mito", "doublet_score", "complexity"]
for metric in qc_metrics:
    sc.pl.violin(adata, metric, groupby="leiden", rotation=90)

### Curated Marker Dotplot

In [ ]:
marker_genes = {
    "Plasma_cell": ["SDC1", "MZB1", "PRDM1", "JCHAIN", "XBP1"],
    "pDC": ["IRF8", "TCF4", "IRF7", "PLAC8"],
    "B_cell": ["MS4A1", "CD79A", "CD74", "HLA-DRA", "CD37"],
    "T_cell": ["CD3D", "CD3E", "IL32"],
    "Myeloid": ["LYZ", "CD14", "S100A8"],
}

sc.pl.dotplot(adata, marker_genes, groupby="leiden", standard_scale="var")
sc.pl.umap(adata, color="leiden", legend_loc="on data")

### Cluster Classification — Review and Edit

Review the dotplot, the unbiased `rank_genes_groups` output, and the
per-cluster QC-metric violins above. Then edit the three lists below so every
cluster this run produced is correctly categorized. The default is
conservative and does **not** reflect a completed review: nothing is flagged
until you edit it.

In [ ]:
# --- EDIT AFTER REVIEWING the evidence above. Every observed cluster ID should be considered. ---
MYELOID_CLUSTERS = []
B_CELL_CLUSTERS = []
QC_CAUTION_CLUSTERS = []
# --- end of editable block ---

adata.obs["cell_type_candidate"] = np.select(
    [
        adata.obs["leiden"].isin(MYELOID_CLUSTERS),
        adata.obs["leiden"].isin(B_CELL_CLUSTERS),
        adata.obs["leiden"].isin(QC_CAUTION_CLUSTERS),
    ],
    ["Myeloid candidate", "B cell candidate", "QC-caution (low counts / high mito)"],
    default="Plasma candidate",
)

sc.pl.umap(adata, color="cell_type_candidate")
adata.obs["cell_type_candidate"].value_counts()

### Sample Composition of the Flagged Clusters

Does a flagged cluster reflect one sample's biology, or a broader batch
effect touching that sample's other clusters too?

In [ ]:
flagged_clusters = sorted(set(MYELOID_CLUSTERS) | set(B_CELL_CLUSTERS) | set(QC_CAUTION_CLUSTERS))
if flagged_clusters:
    flagged_samples = (
        adata.obs.loc[adata.obs["leiden"].isin(flagged_clusters), "sample_id"]
        .value_counts().head(10).index.tolist()
    )
    display(
        pd.crosstab(adata.obs["sample_id"], adata.obs["leiden"], normalize="index")
        .loc[flagged_samples].round(3)
    )
else:
    print("No clusters flagged -- nothing to check here yet.")

## Save Normalized AnnData

In [ ]:
adata.write(OUTPUT_H5AD)

cluster_summary = (
    adata.obs.groupby(["leiden", "clinical_stage_standard"], observed=True)
    .size().rename("n_cells").reset_index()
)
cluster_summary_path = RESULTS_DIR / "preprocessing" / f"{COHORT}_leiden_cluster_summary.csv"
cluster_summary.to_csv(cluster_summary_path, index=False)

n_leiden_clusters = adata.obs["leiden"].nunique()

print(f"Saved normalized AnnData: {OUTPUT_H5AD}")
print(f"Cells: {adata.n_obs:,}  Genes: {adata.n_vars:,}  HVGs: {n_hvg:,}  Leiden clusters: {n_leiden_clusters:,}")
print(f"Saved cluster summary -> {cluster_summary_path}")